# Etapa 1
## Modelos utilzados
1. VGG16
2. ResNet50
3. ConvNeXt-base

## Datasets utilizados
1. CIFAR-100 (60.000 imagens de 100 classes com resolução 32x32): https://www.cs.toronto.edu/~kriz/cifar.html
2. Oxford-IIIT Pet (7.349 imagens de 37 classes de cães e gatos com resolução variável): https://www.robots.ox.ac.uk/~vgg/data/pets/

In [1]:
import torch
import numpy as np
import pandas as pd
import torchvision
from tqdm import tqdm

torch.backends.cudnn.benchmark = True            # autotuner p/ input fixo
torch.backends.cuda.matmul.allow_tf32 = True     # TF32 no Ampere
torch.backends.cudnn.allow_tf32 = True           # TF32 no Ampere
device = torch.device("cuda" if torch.cuda.is_available() else Exception("No GPU available"))


# Baixando os Datasets e carregando em DataLoader

### Baixando o dataset cifar100

In [2]:
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),  
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])

# duas visoes do MESMO conjunto de treino: aug (treino) e limpa (validacao)
cifar_train_full = torchvision.datasets.CIFAR100(root="./data", train=True, download=False, transform=train_transform)
cifar_eval_full  = torchvision.datasets.CIFAR100(root="./data", train=True, download=False, transform=test_transform)

g = torch.Generator()
perm = torch.randperm(len(cifar_train_full), generator=g).tolist()
cut = int(0.9 * len(perm))
train_idx, val_idx = perm[:cut], perm[cut:]

train_set_CIFAR100 = Subset(cifar_train_full, train_idx)   # com augmentation
val_set_CIFAR100   = Subset(cifar_eval_full,  val_idx)     # sem augmentation
test_set_CIFAR100  = torchvision.datasets.CIFAR100(root="./data", train=False, download=False, transform=test_transform)

train_loader_CIFAR100 = DataLoader(train_set_CIFAR100, batch_size=256, shuffle=True,  num_workers=4, pin_memory=True)
val_loader_CIFAR100   = DataLoader(val_set_CIFAR100,   batch_size=256, shuffle=False, num_workers=4, pin_memory=True)
test_loader_CIFAR100  = DataLoader(test_set_CIFAR100,  batch_size=256, shuffle=False, num_workers=4, pin_memory=True)

print(f"CIFAR-100 -> treino {len(train_set_CIFAR100)} | val {len(val_set_CIFAR100)} | teste {len(test_set_CIFAR100)}")

RuntimeError: Dataset not found or corrupted. You can use download=True to download it

### Baixando o dataset oxfordpet

In [3]:
train_transform_pets = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),              
    transforms.RandomHorizontalFlip(p=0.5),  
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform_pets = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224), 
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

oxford_train_full = torchvision.datasets.OxfordIIITPet(root="./data", download=False, transform=train_transform_pets)
oxford_eval_full  = torchvision.datasets.OxfordIIITPet(root="./data", download=False, transform=test_transform_pets)

g = torch.Generator()
perm = torch.randperm(len(oxford_train_full), generator=g).tolist()
cut = int(0.9 * len(perm))
train_idx, val_idx = perm[:cut], perm[cut:]

train_set_oxford102 = Subset(oxford_train_full, train_idx)   # com augmentation
val_set_oxford102   = Subset(oxford_eval_full,  val_idx)     # sem augmentation
test_set_oxford102  = torchvision.datasets.OxfordIIITPet(root="./data", download=False, split="test", transform=test_transform_pets)

train_loader_oxford102 = DataLoader(train_set_oxford102, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader_oxford102   = DataLoader(val_set_oxford102,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader_oxford102  = DataLoader(test_set_oxford102,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Oxford-IIIT Pet -> treino {len(train_set_oxford102)} | val {len(val_set_oxford102)} | teste {len(test_set_oxford102)}")

Oxford-IIIT Pet -> treino 3312 | val 368 | teste 3669


# Carregando os modelos com pesos congelados

In [4]:
vgg_weights = torchvision.models.VGG16_Weights.DEFAULT
vgg_16 = torchvision.models.vgg16(weights=vgg_weights)

resnet_weights = torchvision.models.ResNet50_Weights.DEFAULT
resnet_50 = torchvision.models.resnet50(weights=resnet_weights)

convnext_weights = torchvision.models.ConvNeXt_Base_Weights.DEFAULT
convnext_base = torchvision.models.convnext_base(weights=convnext_weights)

models = {
    "vgg16": vgg_16,
    "resnet50": resnet_50,
    "convnext_base": convnext_base
}

for model in models.values():
    for param in model.parameters():
        param.requires_grad = False


for param in models["vgg16"].classifier.parameters():
    param.requires_grad = True

for param in models["resnet50"].fc.parameters():
    param.requires_grad = True

for param in models["convnext_base"].classifier.parameters():
    param.requires_grad = True


In [5]:
import copy

vgg_16_cifar100 = copy.deepcopy(models["vgg16"])
resnet_50_cifar100 = copy.deepcopy(models["resnet50"])
convnext_base_cifar100 = copy.deepcopy(models["convnext_base"])

vgg_16_cifar100.classifier[6] = torch.nn.Linear(in_features=4096, out_features=100, bias=True)
resnet_50_cifar100.fc = torch.nn.Linear(in_features=2048, out_features=100, bias=True)
convnext_base_cifar100.classifier[2] = torch.nn.Linear(in_features=1024, out_features=100, bias=True)


vgg_16_oxford102 = copy.deepcopy(models["vgg16"])
resnet_50_oxford102 = copy.deepcopy(models["resnet50"])
convnext_base_oxford102 = copy.deepcopy(models["convnext_base"])

vgg_16_oxford102.classifier[6] = torch.nn.Linear(in_features=4096, out_features=37, bias=True)
resnet_50_oxford102.fc = torch.nn.Linear(in_features=2048, out_features=37, bias=True)
convnext_base_oxford102.classifier[2] = torch.nn.Linear(in_features=1024, out_features=37, bias=True)

# Instanciando função de perda e otimizadores

In [6]:
criterion = torch.nn.CrossEntropyLoss()
WEIGHT_DECAY = 1e-2

optimizer_vgg16_cifar100 = torch.optim.AdamW(vgg_16_cifar100.parameters(), lr=0.001, weight_decay=WEIGHT_DECAY)
optimizer_resnet50_cifar100 = torch.optim.AdamW(resnet_50_cifar100.parameters(), lr=0.001, weight_decay=WEIGHT_DECAY)
optimizer_convnext_base_cifar100 = torch.optim.AdamW(convnext_base_cifar100.parameters(), lr=0.001, weight_decay=WEIGHT_DECAY)

models_cifar100 = {
    "vgg16": (vgg_16_cifar100, optimizer_vgg16_cifar100),
    "resnet50": (resnet_50_cifar100, optimizer_resnet50_cifar100),
    "convnext_base": (convnext_base_cifar100, optimizer_convnext_base_cifar100)
}


optimizer_vgg16_oxford102 = torch.optim.AdamW(vgg_16_oxford102.parameters(), lr=0.001, weight_decay=WEIGHT_DECAY)
optimizer_resnet50_oxford102 = torch.optim.AdamW(resnet_50_oxford102.parameters(), lr=0.001, weight_decay=WEIGHT_DECAY)
optimizer_convnext_base_oxford102 = torch.optim.AdamW(convnext_base_oxford102.parameters(), lr=0.001, weight_decay=WEIGHT_DECAY)

models_oxford102 = {
    "vgg16": (vgg_16_oxford102, optimizer_vgg16_oxford102),
    "resnet50": (resnet_50_oxford102, optimizer_resnet50_oxford102),
    "convnext_base": (convnext_base_oxford102, optimizer_convnext_base_oxford102)
}

# Treinamento

In [7]:
@torch.no_grad()
def evaluate_split(model, loader, criterion, device):
    """Loss e acuracia em um conjunto (ex.: validacao). Nao altera pesos."""
    model.eval()
    loss_sum = torch.zeros((), device=device)
    correct  = torch.zeros((), device=device)
    n = 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)
        loss_sum += loss.detach() * images.size(0)
        correct  += (outputs.argmax(1) == labels).sum()
        n += images.size(0)
    return (loss_sum / n).item(), (correct / n).item()


def train_teacher(model, optimizer, train_loader, val_loader, criterion, epochs, save_path, device):
    """Treina validando a cada epoca; salva o checkpoint só quando a loss de validação melhora."""
    model.to(device)
    scaler = torch.amp.GradScaler("cuda")
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    best_val_loss = -1.0

    for epoch in range(epochs):
        model.train()
        run_loss = torch.zeros((), device=device)
        correct  = torch.zeros((), device=device)
        n = 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            with torch.autocast("cuda", dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            run_loss += loss.detach() * images.size(0)
            correct  += (outputs.argmax(1) == labels).sum()
            n += images.size(0)
        scheduler.step()

        train_loss = (run_loss / n).item()
        train_acc  = (correct / n).item()
        val_loss, val_acc = evaluate_split(model, val_loader, criterion, device)

        if val_loss < best_val_loss or best_val_loss == -1:
            best_val_loss = val_loss
            torch.save(model.state_dict(), save_path)
        print(f"epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} | "
              f"validation loss {val_loss:.4f} acc {val_acc:.4f} | lr {scheduler.get_last_lr()[0]:.2e}") 


#### Treinando CIFAR100

In [8]:
NUM_EPOCHS = 25
for name, (model, optimizer) in models_cifar100.items():
    print(f"=== CIFAR-100 | {name} ===")
    train_teacher(model, optimizer, train_loader_CIFAR100, val_loader_CIFAR100,
                  criterion, NUM_EPOCHS, f"./models/{name}_cifar100.pth", device)

=== CIFAR-100 | vgg16 ===
epoch 00 | train loss 2.3257 acc 0.3815 | validation loss 1.5625 acc 0.5592 | lr 9.96e-04
epoch 01 | train loss 1.8657 acc 0.4893 | validation loss 1.4548 acc 0.5972 | lr 9.84e-04
epoch 02 | train loss 1.7711 acc 0.5132 | validation loss 1.4574 acc 0.5990 | lr 9.65e-04
epoch 03 | train loss 1.7210 acc 0.5295 | validation loss 1.4028 acc 0.6138 | lr 9.38e-04
epoch 04 | train loss 1.6411 acc 0.5503 | validation loss 1.4027 acc 0.6182 | lr 9.05e-04
epoch 05 | train loss 1.5977 acc 0.5613 | validation loss 1.3772 acc 0.6300 | lr 8.65e-04
epoch 06 | train loss 1.5407 acc 0.5774 | validation loss 1.3389 acc 0.6362 | lr 8.19e-04
epoch 07 | train loss 1.4698 acc 0.5933 | validation loss 1.3205 acc 0.6472 | lr 7.68e-04
epoch 08 | train loss 1.4132 acc 0.6072 | validation loss 1.2957 acc 0.6482 | lr 7.13e-04
epoch 09 | train loss 1.3462 acc 0.6244 | validation loss 1.2602 acc 0.6572 | lr 6.55e-04
epoch 10 | train loss 1.3046 acc 0.6342 | validation loss 1.2602 acc 0.649

#### Treinando OxfordIIIT Pet

In [9]:
NUM_EPOCHS = 20
for name, (model, optimizer) in models_oxford102.items():
    print(f"=== Oxford-IIIT Pet | {name} ===")
    train_teacher(model, optimizer, train_loader_oxford102, val_loader_oxford102,
                  criterion, NUM_EPOCHS, f"{name}_oxford102.pth", device)

=== Oxford-IIIT Pet | vgg16 ===
epoch 00 | train loss 1.0876 acc 0.7346 | validation loss 0.7495 acc 0.7962 | lr 9.88e-04
epoch 01 | train loss 1.0273 acc 0.7521 | validation loss 0.5559 acc 0.8505 | lr 9.70e-04
epoch 02 | train loss 0.9235 acc 0.7896 | validation loss 0.7473 acc 0.8288 | lr 9.40e-04
epoch 03 | train loss 0.9895 acc 0.7938 | validation loss 0.5531 acc 0.8478 | lr 8.99e-04
epoch 04 | train loss 0.8961 acc 0.8013 | validation loss 0.5892 acc 0.8641 | lr 8.48e-04
epoch 05 | train loss 0.8150 acc 0.8279 | validation loss 0.6329 acc 0.8533 | lr 7.89e-04
epoch 06 | train loss 0.7024 acc 0.8478 | validation loss 0.4427 acc 0.8832 | lr 7.23e-04
epoch 07 | train loss 0.6026 acc 0.8614 | validation loss 0.7236 acc 0.8668 | lr 6.51e-04
epoch 08 | train loss 0.5220 acc 0.8919 | validation loss 0.6860 acc 0.8859 | lr 5.75e-04
epoch 09 | train loss 0.4145 acc 0.9031 | validation loss 0.5255 acc 0.8696 | lr 4.97e-04
epoch 10 | train loss 0.3614 acc 0.9152 | validation loss 0.4284 acc

# Testando os modelos

In [ ]:
@torch.no_grad()
def evaluate(name, model, loader, n_samples, criterion, device):
    model.eval()
    running_loss = torch.zeros((), device=device)
    correct = torch.zeros((), device=device)

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        running_loss += loss.detach() * images.size(0)   # média ponderada
        correct += (outputs.argmax(1) == labels).sum()

    avg_loss = (running_loss / n_samples).item()
    accuracy = (correct / n_samples).item()

    print(f"Model: {name} | Test Loss: {avg_loss:.4f} | Test Acc: {accuracy:.4f}")
    return avg_loss, accuracy


def load_checkpoint(model, path, device):
    """Recarrega os pesos salvos em disco (robusto a crash/restart do kernel)."""
    state_dict = torch.load(path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    return model.to(device)


accuracy = {}

print("=== CIFAR-100 ===")
for name, (model, optimizer) in models_cifar100.items():
    model = load_checkpoint(model, f"./models/{name}_cifar100.pth", device)
    _, test_acc = evaluate(name, model, test_loader_CIFAR100,
                           len(test_set_CIFAR100), criterion, device)
    accuracy[f"{name}_cifar100"] = test_acc

print("=== Oxford-IIIT Pet ===")
for name, (model, optimizer) in models_oxford102.items():
    model = load_checkpoint(model, f"./models/{name}_oxford102.pth", device)
    _, test_acc = evaluate(name, model, test_loader_oxford102,
                           len(test_set_oxford102), criterion, device)
    accuracy[f"{name}_oxford102"] = test_acc

=== CIFAR-100 ===
Model: vgg16 | Test Loss: 1.1118 | Test Acc: 0.6922
Model: resnet50 | Test Loss: 1.4740 | Test Acc: 0.5992
Model: convnext_base | Test Loss: 0.6100 | Test Acc: 0.8217
=== Oxford-IIIT Pet ===
Model: vgg16 | Test Loss: 0.4264 | Test Acc: 0.8893
Model: resnet50 | Test Loss: 0.3589 | Test Acc: 0.9019
Model: convnext_base | Test Loss: 0.1839 | Test Acc: 0.9387


In [11]:
accuracy_csv = pd.DataFrame.from_dict(accuracy, orient="index", columns=["accuracy"])
accuracy_csv.to_csv("./results/teacher_accuracy.csv")

Com o esquema de treino/validação (80/20)+data augmentation+weight_decay=1e-3+epochs=30 encontramos o seguinte resultado:

  Model: vgg16 | Test Loss: 1.2971 | Test Acc: 0.6947

  Model: resnet50 | Test Loss: 1.6339 | Test Acc: 0.5617

  Model: convnext_base | Test Loss: 0.6353 | Test Acc: 0.8121





  Com o esquema de só treino sem validação+sem data augmentation+weight_decay=1e-2+epochs=15:

  Model: vgg16 | Test Loss: 1.4102 | Test Acc: 0.6914

  Model: resnet50 | Test Loss: 1.4784 | Test Acc: 0.5972

  Model: convnext_base | Test Loss: 0.5876 | Test Acc: 0.8240  

